In [ ]:
# %%
# =============================================================================
# kSZ²-21cm : ION_Tvir_MIN SCAN (QMA Style)
# Fixed HII_EFF_FACTOR=10.0, varying ION_Tvir_MIN
# Extended to z=0.0001 for full kSZ integration
# Parallel over ION_Tvir_MIN + Multi-seed support
# =============================================================================

import numpy as np
import matplotlib as mpl
import matplotlib
import matplotlib.pyplot as plt

import py21cmfast as p21c
import os
import glob
import time
from datetime import datetime

# PBS vs desktop backend
if os.environ.get('PBS_JOBID'):
    matplotlib.use('Agg')
    print("✓ Using Agg backend (PBS/server mode)")
else:
    matplotlib.use('Agg')
    print("✓ Using Agg backend")

print(f"py21cmfast version: {p21c.__version__}")

# =============================================================================
# CELL 1a: Output and Cache Directories (QMA Style)
# =============================================================================

plot_dir = "11May2026_kSZ2_21cm_ION_Tvir_scan/plots"

if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
    print(f"Created directory: {plot_dir}")
else:
    print(f"Directory already exists: {plot_dir}")

print(f"All plots will be saved to: {os.path.abspath(plot_dir)}")

# --- Cache directory (PBS-aware + Notebook aware) ---
try:
    # Running as .py script via PBS
    main_cache_dir = os.path.join(
        os.path.dirname(os.path.abspath(__file__)), 
        "20DEC2025_kSZ2_21cm_ION_Tvir_scan", "cache"
    )
except NameError:
    # Running in Jupyter notebook
    main_cache_dir = os.path.join(
        os.getcwd(), 
        "20DEC2025_kSZ2_21cm_ION_Tvir_scan", "cache"
    )

os.makedirs(main_cache_dir, exist_ok=True)
print(f"Cache directory: {main_cache_dir}")

# =============================================================================
# CELL 1b: Global Plot Settings (QMA Style - DO NOT override later)
# =============================================================================

plt.rcParams.update({
    # Font
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 20,
    'axes.labelsize'     : 28,
    'axes.titlesize'     : 22,
    'xtick.labelsize'    : 22,
    'ytick.labelsize'    : 22,
    'legend.fontsize'    : 18,
    'figure.titlesize'   : 20,
    # Ticks
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.major.size'   : 6,
    'ytick.major.size'   : 6,
    'xtick.minor.size'   : 3,
    'ytick.minor.size'   : 3,
    'xtick.major.width'  : 1.0,
    'ytick.major.width'  : 1.0,
    'xtick.minor.width'  : 0.8,
    'ytick.minor.width'  : 0.8,
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    # Lines / axes
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.8,
    'lines.markersize'   : 5,
    # Grid — OFF everywhere
    'axes.grid'          : False,
    # Figure / save
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
})

print("✓ Global plot settings applied (grid OFF, no downstream overrides needed)")

# =============================================================================
# PDF / PNG style contexts + save_pdf_png
# =============================================================================

PDF_STYLE = {
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 28,
    'axes.labelsize'     : 28,
    'axes.titlesize'     : 32,
    'xtick.labelsize'    : 26,
    'ytick.labelsize'    : 26,
    'legend.fontsize'    : 22,
    'figure.titlesize'   : 28,
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.major.size'   : 6,
    'ytick.major.size'   : 6,
    'xtick.minor.size'   : 3,
    'ytick.minor.size'   : 3,
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.8,
    'axes.grid'          : False,
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
}

PNG_STYLE = {
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 16,
    'axes.labelsize'     : 22,
    'axes.titlesize'     : 18,
    'xtick.labelsize'    : 20,
    'ytick.labelsize'    : 20,
    'legend.fontsize'    : 18,
    'figure.titlesize'   : 16,
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.5,
    'axes.grid'          : False,
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
}

def save_pdf_png(plot_func, plot_dir, plot_name, title=None):
    """
    Save a plot as both PDF and PNG using QMA style.
    plot_func(ax) should draw everything except title/grid.
    """
    with mpl.rc_context(PDF_STYLE):
        fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
        plot_func(ax)
        ax.set_title("")
        ax.grid(False)
        fig.savefig(f"{plot_dir}/{plot_name}.pdf")
        plt.close(fig)

    with mpl.rc_context(PNG_STYLE):
        fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
        plot_func(ax)
        ax.grid(False)
        if title is not None:
            ax.set_title(title, fontweight='bold')
        fig.savefig(f"{plot_dir}/{plot_name}.png")
        plt.close(fig)

print("✓ PDF/PNG style contexts + save_pdf_png defined")

# =============================================================================
# CELL 1c: Define Parameters
# =============================================================================

# Test mode for development
TEST_MODE = False

if TEST_MODE:
    print("\n" + "="*70)
    print("🔧 TEST MODE - Small box")
    print("="*70)
    box_len = 100.0
    hii_dim = 32
    n_threads = 8
else:
    print("\n" + "="*70)
    print("🚀 PRODUCTION MODE - Full resolution")
    print("="*70)
    box_len = 800.0
    hii_dim = 128
    n_threads = 16

user_params = p21c.UserParams(
    HII_DIM=hii_dim,
    BOX_LEN=box_len,
    USE_INTERPOLATION_TABLES=True,
    N_THREADS=n_threads
)

z_min = 0.0001
z_max = 20.0

HII_EFF_FACTOR_FIXED = 10.0

# Multi-seed setup (like main QMA code)
RANDOM_SEEDS = list(range(1, 6))   # Start with 5 seeds, increase as needed
N_SEEDS = len(RANDOM_SEEDS)

# ION_Tvir_MIN scan
ION_Tvir_MIN_VALUES = np.linspace(2.0, 5.5, 20)

if TEST_MODE:
    ION_Tvir_MIN_VALUES = ION_Tvir_MIN_VALUES[::5]

print(f"\n=== PARAMETER SCAN SETUP (kSZ²-21cm) ===")
print(f"Fixed HII_EFF_FACTOR = {HII_EFF_FACTOR_FIXED}")
print(f"Scanning {len(ION_Tvir_MIN_VALUES)} values of ION_Tvir_MIN")
print(f"Multi-seed: {N_SEEDS} realisations per ION_Tvir_MIN")
print(f"Total simulations: {len(ION_Tvir_MIN_VALUES) * N_SEEDS}")
print(f"Box: {box_len} Mpc, Resolution: {hii_dim}³")
print(f"z range: {z_min} → {z_max}")

print("\n=== DEFAULT COSMOLOGY ===")
print(p21c.CosmoParams())

print("\n=== DEFAULT ASTROPHYSICS ===")
print(p21c.AstroParams())

print("\n" + "="*70)

/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_cfg.py:57: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_cfg.py:41: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn("Your configuration file is out of date. Updating...")


py21cmfast version: 3.3.1
Created directory: 20DEC2025_kSZ2_21cm_ION_Tvir_scan/plots
All plots will be saved to: /user1/swanith/20DEC2025_kSZ2_21cm_ION_Tvir_scan/plots
✓ Plot settings applied

🚀 RUNNING IN PRODUCTION MODE - FULL RESOLUTION

=== PARAMETER SCAN SETUP (kSZ²-21cm Analysis) ===
Fixed HII_EFF_FACTOR = 10.0
Default ION_Tvir_MIN = 4.70 (log10 K)

Redshift range: z = 0.0001 to 20.0
  (Extended to z~0 for full kSZ integration)

Scanning ION_Tvir_MIN:
  Number of values: 20
  Range (log10 K): 2.00 → 5.50
  Range (linear): 1.00e+02 K → 3.16e+05 K
  Values: [2.         2.18421053 2.36842105 2.55263158 2.73684211 2.92105263
 3.10526316 3.28947368 3.47368421 3.65789474 3.84210526 4.02631579
 4.21052632 4.39473684 4.57894737 4.76315789 4.94736842 5.13157895
 5.31578947 5.5       ]

Total simulations: 20

=== USER PARAMETERS ===
UserParams:
    BOX_LEN                 : 800.0
    DIM                     : 384
    FAST_FCOLL_TABLES       : False
    HII_DIM                 : 128
    HMF

In [ ]:
# %%
# =============================================================================
# CELL 2: Parallel Lightcone Generation — Multi-Seed + ION_Tvir_MIN Scan
# QMA Style: Robust caching, spawn context, progress tracking
# =============================================================================

import time
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

print("\n" + "="*70)
print("RUNNING PARALLEL LIGHTCONE SIMULATIONS (Seeds × ION_Tvir_MIN)")
print("="*70)

# =============================================================================
# Updated Directories (11May2026 as requested)
# =============================================================================
plot_dir = "11May2026_kSZ2_21cm_ION_Tvir_scan/plots"
main_cache_dir = "11May2026_kSZ2_21cm_ION_Tvir_scan/cache"

os.makedirs(plot_dir, exist_ok=True)
os.makedirs(main_cache_dir, exist_ok=True)

print(f"Plots → {os.path.abspath(plot_dir)}")
print(f"Cache → {os.path.abspath(main_cache_dir)}")

# =============================================================================
# Parallel Setup
# =============================================================================
N_TOTAL_CORES = int(os.environ.get('PBS_NCPUS', os.cpu_count() or 16))
N_THREADS_PER_WORKER = 8
N_WORKERS = max(1, N_TOTAL_CORES // N_THREADS_PER_WORKER)
N_WORKERS = min(N_WORKERS, len(ION_Tvir_MIN_VALUES) * N_SEEDS)

print(f"Parallel Execution: {N_WORKERS} workers × {N_THREADS_PER_WORKER} threads")

# =============================================================================
# Worker Function (Top-level for pickling)
# =============================================================================
def _run_or_load_lightcone(seed, tvir, cache_base_dir, z_min, z_max, user_params):
    """Run or load one (seed, ION_Tvir_MIN) combination."""
    import os
    import glob
    import time as _time
    import py21cmfast as _p21c

    # Create unique subdirectory: seed_XXX_TvirX.XXX
    cache_subdir = os.path.join(
        cache_base_dir, 
        f"seed_{seed}_Tvir{tvir:.3f}"
    )
    os.makedirs(cache_subdir, exist_ok=True)

    astro_params = _p21c.AstroParams(
        HII_EFF_FACTOR=HII_EFF_FACTOR_FIXED,
        ION_Tvir_MIN=tvir
    )

    # Cache check
    cached_files = sorted(glob.glob(os.path.join(cache_subdir, "LightCone_*.h5")))
    if cached_files:
        try:
            lc = _p21c.run_lightcone(
                redshift=z_min,
                max_redshift=z_max,
                lightcone_quantities=('brightness_temp', 'density', 'xH_box', 'velocity'),
                user_params=user_params,
                astro_params=astro_params,
                random_seed=seed,
                direc=cache_subdir,
                write=False
            )
            return (seed, tvir, lc, "cached", 0.0)
        except:
            pass  # Fall through to recompute

    # Run new simulation
    sim_start = _time.time()
    try:
        lc = _p21c.run_lightcone(
            redshift=z_min,
            max_redshift=z_max,
            lightcone_quantities=('brightness_temp', 'density', 'xH_box', 'velocity'),
            user_params=user_params,
            astro_params=astro_params,
            random_seed=seed,
            direc=cache_subdir,
            write=True
        )
        sim_time = _time.time() - sim_start
        return (seed, tvir, lc, "computed", sim_time)
    except Exception as e:
        return (seed, tvir, None, f"failed: {e}", _time.time() - sim_start)


# =============================================================================
# Dispatch Parallel Jobs
# =============================================================================
lightcones = {}   # key: (seed, tvir)

scan_start = time.time()
mp_ctx = mp.get_context("spawn")

print(f"\nDispatching {N_SEEDS} seeds × {len(ION_Tvir_MIN_VALUES)} Tvir values "
      f"using {N_WORKERS} workers...")

with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=mp_ctx) as ex:
    futures = {}
    for seed in RANDOM_SEEDS:
        for tvir in ION_Tvir_MIN_VALUES:
            cache_base = os.path.join(main_cache_dir, f"seed_{seed}")
            fut = ex.submit(
                _run_or_load_lightcone,
                seed, tvir, cache_base, z_min, z_max, user_params
            )
            futures[fut] = (seed, tvir)

    completed = 0
    for fut in as_completed(futures):
        seed, tvir, lc, status, sim_time = fut.result()
        key = (seed, tvir)
        lightcones[key] = lc

        completed += 1
        if status == "cached":
            msg = "✓ cached"
        elif status == "computed":
            msg = f"✓ computed ({sim_time/60:.2f} min)"
        else:
            msg = f"✗ {status}"

        elapsed = (time.time() - scan_start) / 60
        print(f"  [{completed:3d}/{len(futures)}] seed={seed:2d}, Tvir={tvir:.3f} → {msg} "
              f"(elapsed: {elapsed:.1f} min)")

total_time = time.time() - scan_start
print(f"\n{'='*70}")
print(f"✓ ALL LIGHTCONES COMPLETE — {total_time/60:.2f} minutes total")
print(f"  Successful: {sum(1 for v in lightcones.values() if v is not None)}/"
      f"{len(lightcones)}")
print(f"{'='*70}")


RUNNING ION_Tvir_MIN SCAN FOR kSZ²-21cm ANALYSIS
Created cache directory: 20DEC2025_kSZ2_21cm_ION_Tvir_scan/cache

SIMULATION 1/20
HII_EFF_FACTOR = 10.0
ION_Tvir_MIN = 2.000 (log10 K) = 1.00e+02 K
Redshift range: z = 0.0001 → 20.0
Box size: 800.0 Mpc
Resolution: 128³ cells


/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:400: UserWarning: The following parameters to FlagOptions are not supported: ['USE_VELS_AUX']
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:815: UserWarning: Trying to remove array that isn't yet created: hires_vx
  warnings.warn(f"Trying to remove array that isn't yet created: {k}")
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:815: UserWarning: Trying to remove array that isn't yet created: hires_vy
  warnings.warn(f"Trying to remove array that isn't yet created: {k}")
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:815: UserWarning: Trying to remove array that isn't yet created: hires_vz
  warnings.warn(f"Trying to remove array that isn't yet created: {k}")
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:815: UserWarning: Trying to r


✓ Simulation complete!
  Time: 2.77 minutes
  Cache: 20DEC2025_kSZ2_21cm_ION_Tvir_scan/cache/HII10_Tvir2.000
  Shape: (128, 128, 1755)
  Redshift range: [0.00, 20.15]
  Quick stats:
    z(10% ionized) = 20.11
    z(50% ionized) = 14.68
    z(90% ionized) = 11.87
    Δz (10%→90%) = 8.24

  Progress: 1/20 (5.0%)
  Average time per sim: 2.92 min
  ETA: 55.4 minutes (~0.92 hours)

SIMULATION 2/20
HII_EFF_FACTOR = 10.0
ION_Tvir_MIN = 2.184 (log10 K) = 1.53e+02 K
Redshift range: z = 0.0001 → 20.0
Box size: 800.0 Mpc
Resolution: 128³ cells

✓ Simulation complete!
  Time: 2.78 minutes
  Cache: 20DEC2025_kSZ2_21cm_ION_Tvir_scan/cache/HII10_Tvir2.184
  Shape: (128, 128, 1755)
  Redshift range: [0.00, 20.15]
  Quick stats:
    z(10% ionized) = 20.11
    z(50% ionized) = 14.38
    z(90% ionized) = 11.37
    Δz (10%→90%) = 8.74

  Progress: 2/20 (10.0%)
  Average time per sim: 2.92 min
  ETA: 52.6 minutes (~0.88 hours)

SIMULATION 3/20
HII_EFF_FACTOR = 10.0
ION_Tvir_MIN = 2.368 (log10 K) = 2.34e+0

In [ ]:
# %%
# =============================================================================
# CELL 4: Reionization History + Optical Depth (Per Seed)
# QMA Style - No early averaging over seeds
# =============================================================================

print("\n" + "="*70)
print("REIONIZATION HISTORY + OPTICAL DEPTH ANALYSIS (Per Seed)")
print("="*70)

# =============================================================================
# Compute per (seed, tvir) 
# =============================================================================
tau_results = {}          # key: (seed, tvir)

for (seed, tvir), lc in lightcones.items():
    if lc is None:
        continue
    
    print(f"  Processing seed={seed}, ION_Tvir_MIN={tvir:.3f}", end="\r")
    
    # Geometry
    red_axis = np.asarray(lc.lightcone_redshifts)
    pos_axis = np.asarray(lc.lightcone_distances)
    ind_z = np.where(red_axis <= z_max)[0]
    red_axis = red_axis[ind_z]
    pos_axis = pos_axis[ind_z]
    
    # Ionization history
    z_nodes = np.asarray(lc.node_redshifts[::-1])
    x_e_nodes = 1.0 - np.asarray(lc.global_xH[::-1])
    
    # Interpolate
    x_e_interp = np.interp(red_axis, z_nodes, x_e_nodes)
    
    # Integration
    ds_Mpc = np.diff(pos_axis)
    z_mid = 0.5 * (red_axis[:-1] + red_axis[1:])
    x_e_mid = 0.5 * (x_e_interp[:-1] + x_e_interp[1:])
    
    dtau = prefactor * x_e_mid * (1.0 + z_mid)**2 * ds_Mpc
    tau = np.cumsum(dtau)
    
    tau_results[(seed, tvir)] = {
        'z_mid': z_mid,
        'tau': tau,
        'tau_total': tau[-1],
        'red_axis': red_axis,
        'z_nodes': z_nodes,
        'x_e_nodes': x_e_nodes
    }

print(f"\n✓ Computed optical depth for {len(tau_results)} realisations "
      f"({N_SEEDS} seeds × {len(ION_Tvir_MIN_VALUES)} ION_Tvir_MIN)")

# =============================================================================
# PLOT 4a: x_e vs z (one line per (seed, tvir))
# =============================================================================
def _draw_xe(ax):
    cmap = mpl.cm.plasma
    norm = mpl.colors.Normalize(vmin=min(ION_Tvir_MIN_VALUES), 
                                vmax=max(ION_Tvir_MIN_VALUES))
    
    for (seed, tvir), data in tau_results.items():
        color = cmap(norm(tvir))
        ax.plot(data['z_nodes'], data['x_e_nodes'], 
                color=color, lw=1.2, alpha=0.6)
    
    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(r'Ionization Fraction $x_e$')
    ax.set_ylim(-0.05, 1.05)
    ax.invert_xaxis()
    
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = ax.figure.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r'ION\_Tvir\_MIN [log$_{10}$K]')

save_pdf_png(
    _draw_xe, plot_dir,
    "reionization_history_xe_ION_Tvir_all",
    title=f'Reionization History: Ionization Fraction (HII_EFF={HII_EFF_FACTOR_FIXED})'
)
print("✓ Saved: reionization_history_xe_ION_Tvir_all")

# =============================================================================
# PLOT 4b: Cumulative τ vs z
# =============================================================================
def _draw_tau(ax):
    cmap = mpl.cm.plasma
    norm = mpl.colors.Normalize(vmin=min(ION_Tvir_MIN_VALUES), 
                                vmax=max(ION_Tvir_MIN_VALUES))
    
    for (seed, tvir), data in tau_results.items():
        color = cmap(norm(tvir))
        ax.plot(data['z_mid'], data['tau'], color=color, lw=1.2, alpha=0.6)
    
    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(r'Cumulative Optical Depth $\tau(<z)$')
    ax.invert_xaxis()
    
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = ax.figure.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r'ION\_Tvir\_MIN [log$_{10}$K]')

save_pdf_png(
    _draw_tau, plot_dir,
    "tau_vs_z_ION_Tvir_all",
    title=f'Cumulative Optical Depth vs Redshift (HII_EFF={HII_EFF_FACTOR_FIXED})'
)
print("✓ Saved: tau_vs_z_ION_Tvir_all")

print("\n✓ CELL 4 COMPLETE")

In [ ]:
# %%
# =============================================================================
# CELL 5: Compute kSZ Integrand with Visibility Function (Per Seed)
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION (Per Seed)")
print("="*70)

c_Mpc_s = 299792.458 / 3.08567758e19
print(f"Speed of light: c = {c_Mpc_s:.6e} Mpc/s")

kSZ_integrand_results = {}   # key: (seed, tvir)

for (seed, tvir), lc in lightcones.items():
    if lc is None:
        continue
    if (seed, tvir) not in tau_results:
        continue
    
    tau_data = tau_results[(seed, tvir)]
    
    # Extract fields
    red_axis_full = np.asarray(lc.lightcone_redshifts, dtype=np.float64)
    ind_z = np.where(red_axis_full <= z_max)[0]
    
    density_1plus = 1 + np.asarray(lc.density[:, :, ind_z])
    x_e_3D        = 1 - np.asarray(lc.xH_box[:, :, ind_z])
    v_los_Mpc_s   = np.asarray(lc.velocity[:, :, ind_z]) / 67.4
    
    # Tau interpolation
    red_axis = np.asarray(tau_data['red_axis'], dtype=np.float64)
    z_mid    = np.asarray(tau_data['z_mid'], dtype=np.float64)
    tau      = np.asarray(tau_data['tau'], dtype=np.float64)
    
    tau_extended = np.concatenate([[0.0], tau])
    z_extended   = np.concatenate([[red_axis[0]], z_mid])
    tau_at_lc    = np.interp(red_axis, z_extended, tau_extended)
    
    visibility = np.exp(-tau_at_lc)
    visibility_3D = visibility[None, None, :]
    
    # Compute integrand
    kSZ_integrand = (density_1plus * x_e_3D * 
                     v_los_Mpc_s / c_Mpc_s * 
                     visibility_3D)
    
    kSZ_integrand_results[(seed, tvir)] = {
        'kSZ_integrand': kSZ_integrand,
        'visibility': visibility,
        'red_axis': red_axis,
        'ind_z': ind_z
    }

print(f"✓ Computed kSZ integrand for {len(kSZ_integrand_results)} realisations")

# =============================================================================
# PLOT 5a: kSZ Integrand (Representative)
# =============================================================================
print("\nGenerating kSZ Integrand Plot...")

tvir_subset = ION_Tvir_MIN_VALUES[::max(1, len(ION_Tvir_MIN_VALUES)//4)]

def _draw_integrand(ax):
    # Pick one representative (seed=1) for each tvir in subset
    for tvir in tvir_subset:
        for s in RANDOM_SEEDS:
            key = (s, tvir)
            if key in kSZ_integrand_results:
                data = kSZ_integrand_results[key]
                break
        else:
            continue
            
        kSZ_integrand = data['kSZ_integrand']
        ind_z = data['ind_z']
        lc = lightcones[(s, tvir)]
        
        slice_2D = kSZ_integrand[:, :, kSZ_integrand.shape[2]//2]
        x_extent = float(np.asarray(lc.lightcone_distances[ind_z].max()))
        y_extent = float(user_params.BOX_LEN)
        
        vmax = float(np.percentile(np.abs(kSZ_integrand), 99))
        
        im = ax.imshow(slice_2D.T, extent=[0, x_extent, 0, y_extent],
                       aspect='auto', cmap='seismic', origin='lower',
                       vmin=-vmax, vmax=vmax)
        
        ax.figure.colorbar(im, ax=ax, fraction=0.046).set_label(
            r'kSZ Integrand')
        ax.set_xlabel('Comoving Distance [Mpc]')
        ax.set_ylabel('Comoving Distance [Mpc]')
        ax.text(0.02, 0.98, f'ION_Tvir_MIN={tvir:.2f} (seed={s})',
                transform=ax.transAxes, fontsize=13, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        break  # plot only one per tvir

save_pdf_png(
    _draw_integrand, plot_dir,
    "kSZ_integrand_with_visibility_ION_Tvir",
    title=r'kSZ Integrand $(1+\delta) x_e v_z/c \, e^{-\tau}$',
    figsize=(13, 8)
)
print("✓ Saved: kSZ_integrand_with_visibility_ION_Tvir")

print("\n✓ CELL 5 COMPLETE")


COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION FOR ALL ION_Tvir_MIN
Speed of light: c = 9.715612e-15 Mpc/s

ION_Tvir_MIN = 2.000 (T = 1.00e+02 K)
3D field shapes: (128, 128, 1753)
τ range: [0.000000, 0.178424]
e^(-τ) range: [0.836587, 1.000000]

kSZ INTEGRAND STATISTICS:
  Mean: -2.7653e-06
  Std:  2.4517e-03
  Min:  -1.0787e-01
  Max:  1.1788e-01
  RMS:  2.4517e-03

ION_Tvir_MIN = 2.184 (T = 1.53e+02 K)
3D field shapes: (128, 128, 1753)
τ range: [0.000000, 0.170248]
e^(-τ) range: [0.843455, 1.000000]

kSZ INTEGRAND STATISTICS:
  Mean: -2.2919e-06
  Std:  2.4320e-03
  Min:  -1.0787e-01
  Max:  1.1788e-01
  RMS:  2.4320e-03

ION_Tvir_MIN = 2.368 (T = 2.34e+02 K)
3D field shapes: (128, 128, 1753)
τ range: [0.000000, 0.161922]
e^(-τ) range: [0.850508, 1.000000]

kSZ INTEGRAND STATISTICS:
  Mean: -1.7067e-06
  Std:  2.4115e-03
  Min:  -1.0787e-01
  Max:  1.1788e-01
  RMS:  2.4115e-03

ION_Tvir_MIN = 2.553 (T = 3.57e+02 K)
3D field shapes: (128, 128, 1753)
τ range: [0.000000, 0.153493]
e

In [ ]:
# %%
# =============================================================================
# CELL 6: Compute kSZ Map at Fixed z_obs = 5.0 (Per Seed)
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ MAPS AT FIXED z_obs = 5.0")
print("="*70)

z_obs = 5.0   # Fixed observation redshift

# Physical constants (CGS)
c_cm_s      = 3.0e10
sigma_T_cm2 = 6.6525e-25
n_e0_cm3    = 2.06e-7
Mpc_to_cm   = 3.0857e24

prefactor_cgs = n_e0_cm3 * sigma_T_cm2 * c_cm_s
print(f"Prefactor = {prefactor_cgs:.4e} s⁻¹")

# Storage: kSZ map at z=5 for each (seed, tvir)
kSZ_maps_at_z5 = {}

scan_start_time = time.time()

for (seed, tvir), lc in lightcones.items():
    if lc is None:
        continue
    if (seed, tvir) not in kSZ_integrand_results:
        continue
    if (seed, tvir) not in tau_results:
        continue

    print(f"  seed={seed:2d} | ION_Tvir_MIN={tvir:.3f} ... ", end="")

    tau_data = tau_results[(seed, tvir)]
    kSZ_data = kSZ_integrand_results[(seed, tvir)]

    # Create cache directory
    kSZ_dir = os.path.join(main_cache_dir, f"seed_{seed}_Tvir{tvir:.3f}", "kSZ_maps")
    os.makedirs(kSZ_dir, exist_ok=True)

    # Extract data
    red_axis     = np.asarray(tau_data['red_axis'], dtype=np.float64)
    z_mid        = np.asarray(tau_data['z_mid'], dtype=np.float64)
    ds_Mpc       = np.diff(np.asarray(lc.lightcone_distances, dtype=np.float64))
    kSZ_integrand = kSZ_data['kSZ_integrand']

    ds_cm = ds_Mpc * Mpc_to_cm

    # Scale factor
    a = 1.0 / (1.0 + red_axis)
    a_squared_mid = 0.5 * (a[:-1]**2 + a[1:]**2)
    a_squared_mid_3D = a_squared_mid[None, None, :]

    kSZ_int_mid = 0.5 * (kSZ_integrand[:, :, :-1] + kSZ_integrand[:, :, 1:])

    kSZ_integrand_full = (prefactor_cgs / a_squared_mid_3D) * \
                         kSZ_int_mid * \
                         (ds_cm / c_cm_s)[None, None, :]

    # ===================================================================
    # Integrate from z_max down to z_obs = 5.0
    # ===================================================================
    idx_integrate = np.where(z_mid >= z_obs)[0]

    if len(idx_integrate) == 0:
        print("No slices beyond z=5")
        continue

    kSZ_map = np.sum(kSZ_integrand_full[:, :, idx_integrate], axis=2)

    # Save
    map_path = os.path.join(kSZ_dir, f"kSZ_map_z{z_obs:.1f}.npy")
    np.save(map_path, kSZ_map)

    # Store in memory
    kSZ_maps_at_z5[(seed, tvir)] = {
        'kSZ_map': kSZ_map,
        'map_path': map_path,
        'z_obs': z_obs
    }

    rms = np.sqrt(np.mean(kSZ_map**2))
    print(f"Done | RMS = {rms:.4e}")

total_time = time.time() - scan_start_time
print(f"\n{'='*70}")
print(f"✓ kSZ MAPS at z={z_obs} COMPLETE")
print(f"Total time: {total_time/60:.2f} minutes")
print(f"Successful maps: {len(kSZ_maps_at_z5)}")
print("="*70)


LINE-OF-SIGHT kSZ MAP INTEGRATION - ION_Tvir_MIN SCAN

=== PHYSICAL CONSTANTS (CGS) ===
c = 3.00e+10 cm/s
σ_T = 6.6525e-25 cm²
n_e0 = 2.0600e-07 cm⁻³
1 Mpc = 3.0857e+24 cm

Prefactor n_e0 × σ_T × c = 4.1112e-21 s⁻¹

ION_Tvir_MIN 1/20: 2.000 (T = 1.00e+02 K)
Number of node redshifts: 155
Redshift range: [0.0001, 20.1090]
kSZ integrand shape: (128, 128, 1752)

=== COMPUTING kSZ MAPS AT EACH NODE REDSHIFT ===
  [  1/155] z=0.0001: RMS=1.8074e-05, Mean=-3.7012e-06 | ETA: 13.7s
  [ 10/155] z=0.1952: RMS=1.8049e-05, Mean=-3.7068e-06 | ETA: 12.4s
  [ 20/155] z=0.4570: RMS=1.8005e-05, Mean=-3.7198e-06 | ETA: 11.0s
  [ 30/155] z=0.7760: RMS=1.7897e-05, Mean=-3.6861e-06 | ETA: 9.7s
  [ 40/155] z=1.1650: RMS=1.7613e-05, Mean=-3.6314e-06 | ETA: 8.4s
  [ 50/155] z=1.6391: RMS=1.7482e-05, Mean=-3.6415e-06 | ETA: 7.3s
  [ 60/155] z=2.2170: RMS=1.7297e-05, Mean=-3.8185e-06 | ETA: 6.2s
  [ 70/155] z=2.9215: RMS=1.7020e-05, Mean=-3.9293e-06 | ETA: 5.1s
  [ 80/155] z=3.7803: RMS=1.6511e-05, Mean=-3.7629

In [ ]:
# %%
# =============================================================================
# CELL 7: kSZ²-21cm Cross-Correlation Power Spectra at z_obs = 5.0
# Per (seed, tvir) — Proper error budget (sample + cosmic variance)
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA (z_obs=5.0)")
print("="*70)

# =============================================================================
# Map & k-space Setup (Common to All)
# =============================================================================
npix_side    = user_params.HII_DIM
box_size_Mpc = float(user_params.BOX_LEN)
pix_size_Mpc = box_size_Mpc / npix_side
pix_area     = pix_size_Mpc**2

dk = 2 * np.pi / (npix_side * pix_size_Mpc)
kx = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
ky = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
kgrid = np.sqrt(kx[:, None]**2 + ky[None, :]**2)

k_bins    = np.logspace(np.log10(dk), np.log10(kgrid.max() * 0.9), 35)
k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])

print(f"Map size: {npix_side}×{npix_side} pixels")
print(f"k-space: dk = {dk:.5f} Mpc⁻¹, {len(k_centers)} bins")

# =============================================================================
# Storage
# =============================================================================
cross_corr_all = {}   # key: (seed, tvir)

scan_start_time = time.time()

for (seed, tvir), lc in lightcones.items():
    if lc is None:
        continue
    if (seed, tvir) not in kSZ_maps_at_z5:
        continue

    print(f"\n{'='*70}")
    print(f"Cross-correlation | seed={seed:2d} | ION_Tvir_MIN={tvir:.3f}")
    print(f"{'='*70}")

    # Load kSZ map at z=5.0
    kSZ_info = kSZ_maps_at_z5[(seed, tvir)]
    kSZ_map = kSZ_info['kSZ_map']

    # Square it
    kSZ2_map = kSZ_map**2
    kSZ2_centered = kSZ2_map - np.mean(kSZ2_map)
    fft_kSZ2_shifted = np.fft.fftshift(np.fft.fft2(kSZ2_centered))

    # Get 21cm slices up to z=5
    lc_redshifts = np.asarray(lc.lightcone_redshifts, dtype=np.float64)
    valid_idx = np.where(lc_redshifts <= 5.0)[0]

    cross_corr_results = {}
    loop_start = time.time()

    for i, idx in enumerate(valid_idx):
        z_21cm = lc_redshifts[idx]

        T21_slice = np.asarray(lc.brightness_temp[:, :, idx])
        T21_centered = T21_slice - np.mean(T21_slice)
        fft_T21_shifted = np.fft.fftshift(np.fft.fft2(T21_centered))

        # Cross-power 2D
        cross_ps2d = np.real(np.conj(fft_kSZ2_shifted) * fft_T21_shifted) \
                     * pix_area / (npix_side**4)

        # Auto-powers
        auto_kSZ2_ps2d = np.abs(fft_kSZ2_shifted)**2 * pix_area / (npix_side**4)
        auto_T21_ps2d  = np.abs(fft_T21_shifted)**2  * pix_area / (npix_side**4)

        # ===================================================================
        # 1D Binning with Full Error Budget
        # ===================================================================
        C_cross_1d = np.zeros(len(k_centers))
        err_sample = np.zeros(len(k_centers))
        err_cosmic = np.zeros(len(k_centers))
        err_total  = np.zeros(len(k_centers))
        P_kSZ2_1d  = np.zeros(len(k_centers))
        P_T21_1d   = np.zeros(len(k_centers))

        for j in range(len(k_centers)):
            mask = (kgrid >= k_bins[j]) & (kgrid < k_bins[j+1])
            n_pix = np.sum(mask)

            if n_pix > 0:
                cross_vals = cross_ps2d[mask]
                
                C_cross_1d[j] = np.mean(cross_vals)
                
                # Sample variance
                err_sample[j] = np.std(cross_vals) / np.sqrt(n_pix)
                
                # Cosmic variance
                Pk1 = np.mean(auto_kSZ2_ps2d[mask])
                Pk2 = np.mean(auto_T21_ps2d[mask])
                err_cosmic[j] = np.sqrt(Pk1 * Pk2 + C_cross_1d[j]**2) / np.sqrt(n_pix)
                
                # Total error
                err_total[j] = np.sqrt(err_sample[j]**2 + err_cosmic[j]**2)
                
                P_kSZ2_1d[j] = Pk1
                P_T21_1d[j]  = Pk2

        cross_corr_results[z_21cm] = {
            'k_centers': k_centers,
            'C_cross_1d': C_cross_1d,
            'C_cross_err_sample': err_sample,
            'C_cross_err_cosmic': err_cosmic,
            'C_cross_err_total':  err_total,
            'P_kSZ2_1d': P_kSZ2_1d,
            'P_T21_1d': P_T21_1d,
            'z_actual': float(z_21cm),
            'kSZ2_rms': float(np.sqrt(np.mean(kSZ2_map**2))),
            'T21_rms': float(np.sqrt(np.mean(T21_slice**2))),
            'T21_mean': float(np.mean(T21_slice))
        }

        if (i + 1) % 8 == 0 or i == 0 or i == len(valid_idx)-1:
            elapsed = time.time() - loop_start
            eta = (elapsed / (i+1)) * (len(valid_idx) - i - 1)
            sign = "+" if np.nanmean(C_cross_1d) > 0 else "-"
            print(f"  [{i+1:3d}/{len(valid_idx)}] z={z_21cm:.3f} | "
                  f"sign={sign} | ETA: {eta:.1f}s")

    loop_time = time.time() - loop_start

    cross_corr_all[(seed, tvir)] = {
        'cross_corr_results': cross_corr_results,
        'computation_time': loop_time,
        'z_obs': 5.0
    }

    print(f"✓ Completed seed={seed}, Tvir={tvir:.3f} — {len(cross_corr_results)} redshifts")

total_time = time.time() - scan_start_time
print(f"\n{'='*70}")
print(f"ALL CROSS-CORRELATIONS COMPLETE — {total_time/60:.2f} minutes")
print(f"Processed {len(cross_corr_all)} realisations")
print("="*70)


COMPUTING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA - ION_Tvir_MIN SCAN

=== MAP PROPERTIES ===
Map size: 128 × 128 pixels
Physical size: 800.0 × 800.0 Mpc²
Pixel size: 6.250 Mpc/pixel

=== k-SPACE GRID ===
dk (fundamental): 0.007854 Mpc⁻¹
k range: [0.000000, 0.710861] Mpc⁻¹

ION_Tvir_MIN 1/20: 2.000 (T = 1.00e+02 K)
Number of node redshifts: 155
Redshift range: [0.0001, 20.1090]

=== COMPUTING CROSS-CORRELATIONS ===
  [  1/155] z=0.0001: Sign=-, T21_mean=0.00 mK | ETA: 694.1s
  [ 10/155] z=0.1952: Sign=-, T21_mean=0.00 mK | ETA: 644.3s
  [ 20/155] z=0.4570: Sign=-, T21_mean=0.00 mK | ETA: 600.8s
  [ 30/155] z=0.7760: Sign=-, T21_mean=0.00 mK | ETA: 557.2s
  [ 40/155] z=1.1650: Sign=-, T21_mean=0.00 mK | ETA: 512.8s
  [ 50/155] z=1.6391: Sign=-, T21_mean=0.00 mK | ETA: 468.3s
  [ 60/155] z=2.2170: Sign=-, T21_mean=0.00 mK | ETA: 423.7s
  [ 70/155] z=2.9215: Sign=-, T21_mean=0.00 mK | ETA: 379.1s
  [ 80/155] z=3.7803: Sign=-, T21_mean=0.00 mK | ETA: 334.5s
  [ 90/155] z=4.8272: Sign=-,

In [ ]:
# =============================================================================
# NOT FOR REPORT
# PLOT: kSZ, kSZ², and 21cm Maps Side-by-Side for Selected ION_Tvir_MIN
# Fixed x_e ~ 0.5, varying ION_Tvir_MIN
# =============================================================================

print(f"\n=== PLOTTING kSZ vs kSZ² vs 21cm MAPS FOR ION_Tvir_MIN SCAN ===")

# Target ionization fraction for comparison
target_xe = 0.5

# Select ION_Tvir_MIN values to plot
tvir_to_plot = tvir_subset  # Use the subset defined earlier

print(f"Plotting maps for {len(tvir_to_plot)} ION_Tvir_MIN values at x_e ~ {target_xe}")

# Create figure: rows = ION_Tvir_MIN, cols = [kSZ, kSZ², 21cm]
fig, axes = plt.subplots(len(tvir_to_plot), 3, 
                         figsize=(16, 5*len(tvir_to_plot)), 
                         constrained_layout=True)

if len(tvir_to_plot) == 1:
    axes = axes.reshape(1, -1)

for row_idx, tvir in enumerate(tvir_to_plot):
    
    if tvir not in lightcones or tvir not in cross_corr_all_tvir:
        # Fill with empty axes
        for col_idx in range(3):
            axes[row_idx, col_idx].text(0.5, 0.5, 'No data', 
                                        ha='center', va='center',
                                        transform=axes[row_idx, col_idx].transAxes,
                                        fontsize=16)
        continue
    
    lightcone = lightcones[tvir]
    kSZ_data = kSZ_maps_all_tvir[tvir]
    kSZ_maps_dir = kSZ_data['kSZ_maps_dir']
    
    # Find redshift closest to target x_e for this ION_Tvir_MIN
    z_nodes_sorted = np.asarray(lightcone.node_redshifts[::-1])
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    idx_xe = np.argmin(np.abs(x_e_nodes - target_xe))
    z_obs = z_nodes_sorted[idx_xe]
    x_e = x_e_nodes[idx_xe]
    
    # Load kSZ map
    kSZ_map_file = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.6f}.npy"
    
    if not os.path.exists(kSZ_map_file):
        for col_idx in range(3):
            axes[row_idx, col_idx].text(0.5, 0.5, f'Map not found\nz={z_obs:.2f}', 
                                        ha='center', va='center',
                                        transform=axes[row_idx, col_idx].transAxes,
                                        fontsize=14)
        continue
    
    kSZ_map = np.load(kSZ_map_file)
    
    # Square it
    kSZ2_map = kSZ_map**2
    
    # Get lightcone redshift axis
    lc_redshifts = np.asarray(lightcone.lightcone_redshifts, dtype=np.float64)
    
    # Find closest lightcone slice to z_obs
    idx_closest = np.argmin(np.abs(lc_redshifts - z_obs))
    z_actual = lc_redshifts[idx_closest]
    
    # Extract 21cm brightness temperature slice
    T21_slice = np.asarray(lightcone.brightness_temp[:, :, idx_closest])
    
    # =============================================================================
    # Left panel: kSZ map
    # =============================================================================
    
    ax_kSZ = axes[row_idx, 0]
    
    # Symmetric color scale for kSZ
    vmax_kSZ = np.percentile(np.abs(kSZ_map), 99)
    
    im_kSZ = ax_kSZ.imshow(kSZ_map.T,
                           cmap='seismic',
                           origin='lower',
                           extent=[0, box_size_Mpc, 0, box_size_Mpc],
                           aspect='equal',
                           vmin=-vmax_kSZ,
                           vmax=vmax_kSZ)
    
    # Colorbar
    cbar_kSZ = plt.colorbar(im_kSZ, ax=ax_kSZ, fraction=0.046, pad=0.04)
    cbar_kSZ.set_label('kSZ', fontsize=11)
    cbar_kSZ.ax.tick_params(labelsize=9)
    
    # Labels
    ax_kSZ.set_xlabel('x [Mpc]', fontsize=12)
    ax_kSZ.set_ylabel('y [Mpc]', fontsize=12)
    ax_kSZ.set_title(f'kSZ Map\nTvir={tvir:.2f} (T={10**tvir:.1e}K)', 
                     fontsize=12, fontweight='bold')
    
    # Stats
    rms_kSZ = np.sqrt(np.mean(kSZ_map**2))
    ax_kSZ.text(0.05, 0.95, 
               f'z={z_obs:.2f}, $x_e$={x_e:.2f}\nRMS={rms_kSZ:.2e}',
               transform=ax_kSZ.transAxes, fontsize=10,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # =============================================================================
    # Middle panel: kSZ² map
    # =============================================================================
    
    ax_kSZ2 = axes[row_idx, 1]
    
    # Use 'hot' colormap for squared map (all positive)
    vmax_kSZ2 = np.percentile(kSZ2_map, 99)
    
    im_kSZ2 = ax_kSZ2.imshow(kSZ2_map.T,
                             cmap='hot',
                             origin='lower',
                             extent=[0, box_size_Mpc, 0, box_size_Mpc],
                             aspect='equal',
                             vmin=0,
                             vmax=vmax_kSZ2)
    
    # Colorbar
    cbar_kSZ2 = plt.colorbar(im_kSZ2, ax=ax_kSZ2, fraction=0.046, pad=0.04)
    cbar_kSZ2.set_label('kSZ²', fontsize=11)
    cbar_kSZ2.ax.tick_params(labelsize=9)
    
    # Labels
    ax_kSZ2.set_xlabel('x [Mpc]', fontsize=12)
    ax_kSZ2.set_ylabel('y [Mpc]', fontsize=12)
    ax_kSZ2.set_title(f'kSZ² Map\nTvir={tvir:.2f} (T={10**tvir:.1e}K)', 
                      fontsize=12, fontweight='bold')
    
    # Stats
    rms_kSZ2 = np.sqrt(np.mean(kSZ2_map**2))
    ax_kSZ2.text(0.05, 0.95, 
                f'z={z_obs:.2f}, $x_e$={x_e:.2f}\nRMS={rms_kSZ2:.2e}',
                transform=ax_kSZ2.transAxes, fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # =============================================================================
    # Right panel: 21cm brightness temperature map
    # =============================================================================
    
    ax_T21 = axes[row_idx, 2]
    
    # Use 'RdBu_r' for 21cm
    vmax_T21 = np.percentile(np.abs(T21_slice), 99)
    
    im_T21 = ax_T21.imshow(T21_slice.T,
                           cmap='RdBu_r',
                           origin='lower',
                           extent=[0, box_size_Mpc, 0, box_size_Mpc],
                           aspect='equal',
                           vmin=-vmax_T21,
                           vmax=vmax_T21)
    
    # Colorbar
    cbar_T21 = plt.colorbar(im_T21, ax=ax_T21, fraction=0.046, pad=0.04)
    cbar_T21.set_label('21cm [mK]', fontsize=11)
    cbar_T21.ax.tick_params(labelsize=9)
    
    # Labels
    ax_T21.set_xlabel('x [Mpc]', fontsize=12)
    ax_T21.set_ylabel('y [Mpc]', fontsize=12)
    ax_T21.set_title(f'21cm Map\nTvir={tvir:.2f} (T={10**tvir:.1e}K)', 
                     fontsize=12, fontweight='bold')
    
    # Stats
    rms_T21 = np.sqrt(np.mean(T21_slice**2))
    ax_T21.text(0.05, 0.95, 
               f'z={z_actual:.2f}, $x_e$={x_e:.2f}\nRMS={rms_T21:.1f}mK',
               transform=ax_T21.transAxes, fontsize=10,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Overall title
fig.suptitle(f'Maps at $x_e$ ~ {target_xe} for Different ION_Tvir_MIN (HII_EFF={HII_EFF_FACTOR_FIXED})', 
            fontsize=20, fontweight='bold', y=0.995)

# Save
plot_name = "kSZ_kSZ2_21cm_maps_ION_Tvir_comparison"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n✓ MAP COMPARISON PLOTTING COMPLETE!")


=== PLOTTING kSZ vs kSZ² vs 21cm MAPS FOR ION_Tvir_MIN SCAN ===
Plotting maps for 4 ION_Tvir_MIN values at x_e ~ 0.5
✓ Saved: kSZ_kSZ2_21cm_maps_ION_Tvir_comparison

✓ MAP COMPARISON PLOTTING COMPLETE!


In [ ]:
# =============================================================================
# NOT FOR REPORT
# DIAGNOSTIC PLOTS: 2D FFT, Power Spectra vs k - ION_Tvir_MIN Scan
# =============================================================================

print(f"\n{'='*70}")
print("GENERATING DIAGNOSTIC PLOTS FOR ION_Tvir_MIN SCAN")
print(f"{'='*70}")

# =============================================================================
# PLOT 1: 2D FFT Maps (k-space) for Selected ION_Tvir_MIN at x_e ~ 0.5
# =============================================================================

print(f"\n=== PLOTTING 2D FFT MAPS IN k-SPACE ===")

# Target ionization fraction for comparison
target_xe = 0.5

# Select ION_Tvir_MIN values to plot
tvir_to_plot = tvir_subset

print(f"Plotting 2D FFT for {len(tvir_to_plot)} ION_Tvir_MIN values at x_e ~ {target_xe}")

# Create figure: rows = ION_Tvir_MIN, cols = [kSZ² FFT, 21cm FFT]
fig, axes = plt.subplots(len(tvir_to_plot), 2, 
                         figsize=(14, 6*len(tvir_to_plot)), 
                         constrained_layout=True)

if len(tvir_to_plot) == 1:
    axes = axes.reshape(1, -1)

for row_idx, tvir in enumerate(tvir_to_plot):
    
    if tvir not in lightcones or tvir not in cross_corr_all_tvir:
        for col_idx in range(2):
            axes[row_idx, col_idx].text(0.5, 0.5, 'No data', 
                                        ha='center', va='center',
                                        transform=axes[row_idx, col_idx].transAxes,
                                        fontsize=16)
        continue
    
    lightcone = lightcones[tvir]
    kSZ_data = kSZ_maps_all_tvir[tvir]
    kSZ_maps_dir = kSZ_data['kSZ_maps_dir']
    
    # Find redshift closest to target x_e
    z_nodes_sorted = np.asarray(lightcone.node_redshifts[::-1])
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    idx_xe = np.argmin(np.abs(x_e_nodes - target_xe))
    z_obs = z_nodes_sorted[idx_xe]
    x_e = x_e_nodes[idx_xe]
    
    # Load kSZ map
    kSZ_map_file = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.6f}.npy"
    
    if not os.path.exists(kSZ_map_file):
        for col_idx in range(2):
            axes[row_idx, col_idx].text(0.5, 0.5, 'Map not found', 
                                        ha='center', va='center',
                                        transform=axes[row_idx, col_idx].transAxes,
                                        fontsize=14)
        continue
    
    kSZ_map = np.load(kSZ_map_file)
    kSZ2_map = kSZ_map**2
    kSZ2_map_centered = kSZ2_map - np.mean(kSZ2_map)
    
    # Get 21cm slice
    lc_redshifts = np.asarray(lightcone.lightcone_redshifts, dtype=np.float64)
    idx_closest = np.argmin(np.abs(lc_redshifts - z_obs))
    T21_slice = np.asarray(lightcone.brightness_temp[:, :, idx_closest])
    T21_slice_centered = T21_slice - np.mean(T21_slice)
    
    # Compute FFTs
    fft_kSZ2 = np.fft.fft2(kSZ2_map_centered)
    fft_kSZ2_shifted = np.fft.fftshift(fft_kSZ2)
    
    fft_T21 = np.fft.fft2(T21_slice_centered)
    fft_T21_shifted = np.fft.fftshift(fft_T21)
    
    # k-space extent
    k_max = kgrid.max()
    
    # =============================================================================
    # Left panel: kSZ² FFT
    # =============================================================================
    
    ax_fft_kSZ2 = axes[row_idx, 0]
    
    power_kSZ2 = np.abs(fft_kSZ2_shifted)**2
    power_kSZ2_log = np.log10(power_kSZ2 + 1e-20)
    
    im_fft_kSZ2 = ax_fft_kSZ2.imshow(power_kSZ2_log.T,
                                      cmap='viridis',
                                      origin='lower',
                                      extent=[-k_max, k_max, -k_max, k_max],
                                      aspect='equal')
    
    cbar_fft_kSZ2 = plt.colorbar(im_fft_kSZ2, ax=ax_fft_kSZ2, fraction=0.046, pad=0.04)
    cbar_fft_kSZ2.set_label(r'log$_{10}$(Power)', fontsize=11)
    cbar_fft_kSZ2.ax.tick_params(labelsize=9)
    
    ax_fft_kSZ2.set_xlabel(r'$k_x$ [Mpc$^{-1}$]', fontsize=12)
    ax_fft_kSZ2.set_ylabel(r'$k_y$ [Mpc$^{-1}$]', fontsize=12)
    ax_fft_kSZ2.set_title(f'kSZ² Power (k-space)\nTvir={tvir:.2f}, z={z_obs:.2f}, $x_e$={x_e:.2f}', 
                          fontsize=12, fontweight='bold')
    
    # Reference circle
    circle = plt.Circle((0, 0), 0.1, color='white', fill=False, linestyle='--', linewidth=1.5)
    ax_fft_kSZ2.add_patch(circle)
    
    # =============================================================================
    # Right panel: 21cm FFT
    # =============================================================================
    
    ax_fft_T21 = axes[row_idx, 1]
    
    power_T21 = np.abs(fft_T21_shifted)**2
    power_T21_log = np.log10(power_T21 + 1e-20)
    
    im_fft_T21 = ax_fft_T21.imshow(power_T21_log.T,
                                    cmap='viridis',
                                    origin='lower',
                                    extent=[-k_max, k_max, -k_max, k_max],
                                    aspect='equal')
    
    cbar_fft_T21 = plt.colorbar(im_fft_T21, ax=ax_fft_T21, fraction=0.046, pad=0.04)
    cbar_fft_T21.set_label(r'log$_{10}$(Power)', fontsize=11)
    cbar_fft_T21.ax.tick_params(labelsize=9)
    
    ax_fft_T21.set_xlabel(r'$k_x$ [Mpc$^{-1}$]', fontsize=12)
    ax_fft_T21.set_ylabel(r'$k_y$ [Mpc$^{-1}$]', fontsize=12)
    ax_fft_T21.set_title(f'21cm Power (k-space)\nTvir={tvir:.2f}, z={z_obs:.2f}, $x_e$={x_e:.2f}', 
                         fontsize=12, fontweight='bold')
    
    # Reference circle
    circle = plt.Circle((0, 0), 0.1, color='white', fill=False, linestyle='--', linewidth=1.5)
    ax_fft_T21.add_patch(circle)

fig.suptitle(f'2D Power Spectra at $x_e$ ~ {target_xe} (HII_EFF={HII_EFF_FACTOR_FIXED})', 
            fontsize=20, fontweight='bold', y=0.995)

plot_name = "2D_FFT_kspace_kSZ2_21cm_ION_Tvir"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 2: Cross-Power vs k for Selected ION_Tvir_MIN at x_e ~ 0.5
# =============================================================================

print(f"\n=== PLOTTING CROSS-POWER vs k ===")

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

# Create colormap for ION_Tvir_MIN
cmap = mpl.cm.plasma
norm = mpl.colors.Normalize(vmin=ION_Tvir_MIN_VALUES.min(), vmax=ION_Tvir_MIN_VALUES.max())

for tvir in ION_Tvir_MIN_VALUES:
    
    if tvir not in cross_corr_all_tvir:
        continue
    
    lightcone = lightcones[tvir]
    cross_corr_results = cross_corr_all_tvir[tvir]['cross_corr_results']
    
    # Find redshift closest to target x_e
    z_nodes_sorted = np.asarray(lightcone.node_redshifts[::-1])
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    idx_xe = np.argmin(np.abs(x_e_nodes - target_xe))
    z_obs = z_nodes_sorted[idx_xe]
    
    if z_obs not in cross_corr_results:
        continue
    
    results = cross_corr_results[z_obs]
    k_centers = results['k_centers']
    C_cross = results['C_cross_1d']
    
    valid = ~np.isnan(C_cross) & np.isfinite(C_cross)
    
    if np.sum(valid) > 5:
        color = cmap(norm(tvir))
        ax.plot(k_centers[valid], C_cross[valid], 
               color=color, linewidth=2, alpha=0.7,
               marker='o', markersize=3)

ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=18)
ax.set_ylabel(r'Cross-Power [Mpc$^2$]', fontsize=18)
ax.set_xscale('log')
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.grid(True, alpha=0.3)

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'ION\_Tvir\_MIN [log$_{10}$K]', fontsize=16)

ax.set_title(f'kSZ²-21cm Cross-Power at $x_e$ ~ {target_xe} (HII_EFF={HII_EFF_FACTOR_FIXED})', 
            fontsize=18, fontweight='bold')

plot_name = "cross_power_vs_k_ION_Tvir_xe05"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 3: Cross-Power vs k for Single ION_Tvir_MIN, Multiple Redshifts
# =============================================================================

print(f"\n=== PLOTTING CROSS-POWER EVOLUTION FOR SINGLE ION_Tvir_MIN ===")

# Pick middle ION_Tvir_MIN value
tvir_middle = sorted(cross_corr_all_tvir.keys())[len(cross_corr_all_tvir)//2]

if tvir_middle in cross_corr_all_tvir:
    
    cross_corr_results = cross_corr_all_tvir[tvir_middle]['cross_corr_results']
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)
    
    # Sample every 15th redshift
    z_sample = sorted(cross_corr_results.keys())[::15]
    
    cmap_z = mpl.cm.rainbow
    norm_z = mpl.colors.Normalize(vmin=min(z_sample), vmax=max(z_sample))
    
    for z_obs in z_sample:
        results = cross_corr_results[z_obs]
        k_centers = results['k_centers']
        C_cross = results['C_cross_1d']
        
        valid = ~np.isnan(C_cross) & np.isfinite(C_cross)
        
        if np.sum(valid) > 5:
            color = cmap_z(norm_z(z_obs))
            ax.plot(k_centers[valid], C_cross[valid], 
                   color=color, linewidth=2, alpha=0.7,
                   marker='o', markersize=3)
    
    ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=18)
    ax.set_ylabel(r'Cross-Power [Mpc$^2$]', fontsize=18)
    ax.set_xscale('log')
    ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.grid(True, alpha=0.3)
    
    # Add colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap_z, norm=norm_z)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r'Redshift $z$', fontsize=16)
    
    ax.set_title(f'Cross-Power Evolution (Tvir={tvir_middle:.2f}, HII_EFF={HII_EFF_FACTOR_FIXED})', 
                fontsize=18, fontweight='bold')
    
    plot_name = f"cross_power_vs_k_z_evolution_Tvir{tvir_middle:.2f}"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

print("\n✓ ALL DIAGNOSTIC PLOTTING COMPLETE!")


GENERATING DIAGNOSTIC PLOTS FOR ION_Tvir_MIN SCAN

=== PLOTTING 2D FFT MAPS IN k-SPACE ===
Plotting 2D FFT for 4 ION_Tvir_MIN values at x_e ~ 0.5
✓ Saved: 2D_FFT_kspace_kSZ2_21cm_ION_Tvir

=== PLOTTING CROSS-POWER vs k ===
✓ Saved: cross_power_vs_k_ION_Tvir_xe05

=== PLOTTING CROSS-POWER EVOLUTION FOR SINGLE ION_Tvir_MIN ===
✓ Saved: cross_power_vs_k_z_evolution_Tvir3.84

✓ ALL DIAGNOSTIC PLOTTING COMPLETE!


In [ ]:
# %%
# =============================================================================
# CELL 8: Final Visualization — D_ℓ(ℓ=3000) + Reionization History + Milestones
# Three-panel plot (Main Science Plot)
# =============================================================================

print("\n" + "="*70)
print("FINAL VISUALIZATION: D_ℓ(ℓ=3000) + Reionization History + Milestones")
print("="*70)

plot_dir_final = os.path.join(plot_dir, "plot_final_cell")
os.makedirs(plot_dir_final, exist_ok=True)
plot_dir_save = plot_dir_final

from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086)
T_CMB_0_uK = 2.725 * 1e6

# =============================================================================
# Convert cross-power to ℓ-space with full error propagation
# =============================================================================
cross_corr_ell_all = {}

for (seed, tvir), data in cross_corr_all.items():
    cross_corr_results = data.get('cross_corr_results', {})
    ell_results = {}
    
    for z_obs, res in cross_corr_results.items():
        D_A_Mpc = float(cosmo.comoving_transverse_distance(z_obs).value)
        k_centers = res['k_centers']
        ell = k_centers * D_A_Mpc
        
        C_ell = res['C_cross_1d'] / D_A_Mpc**2
        err_total = res.get('C_cross_err_total', res.get('C_cross_err', np.zeros_like(C_ell))) / D_A_Mpc**2
        
        D_ell = ell * (ell + 1) * C_ell / (2 * np.pi)
        D_ell_err = ell * (ell + 1) * err_total / (2 * np.pi)
        
        D_ell_uK2_mK = D_ell * T_CMB_0_uK**2
        D_ell_err_uK2_mK = D_ell_err * T_CMB_0_uK**2
        
        ell_results[z_obs] = {
            'ell': ell,
            'D_ell_uK2_mK': D_ell_uK2_mK,
            'D_ell_err_uK2_mK': D_ell_err_uK2_mK,
            'z_actual': z_obs
        }
    
    cross_corr_ell_all[(seed, tvir)] = ell_results

print(f"✓ Converted to ℓ-space for {len(cross_corr_ell_all)} realisations")

# =============================================================================
# Aggregate over seeds per ION_Tvir_MIN
# =============================================================================
from collections import defaultdict
agg_data = defaultdict(list)

for (seed, tvir), res_dict in cross_corr_ell_all.items():
    for z, vals in res_dict.items():
        agg_data[(tvir, z)].append({
            'D_ell': vals['D_ell_uK2_mK'],
            'D_err': vals['D_ell_err_uK2_mK']
        })

final_stats = {}
for (tvir, z), lst in agg_data.items():
    D_vals = np.array([d['D_ell'] for d in lst])
    err_vals = np.array([d['D_err'] for d in lst])
    
    D_mean = np.mean(D_vals)
    D_seed_std = np.std(D_vals, ddof=1) if len(D_vals) > 1 else 0.0
    D_meas_err = np.mean(err_vals)
    D_total_err = np.sqrt(D_seed_std**2 + D_meas_err**2)
    
    final_stats[(tvir, z)] = {
        'D_mean': D_mean,
        'D_total_err': D_total_err
    }

# =============================================================================
# THREE-PANEL FINAL PLOT
# =============================================================================
fig = plt.figure(figsize=(20, 7), constrained_layout=True)
gs = fig.add_gridspec(1, 3)

ax_dl   = fig.add_subplot(gs[0, 0])   # D_ℓ at ℓ=3000
ax_xe   = fig.add_subplot(gs[0, 1])   # Reionization histories
ax_mil  = fig.add_subplot(gs[0, 2])   # Milestones

cmap = mpl.cm.plasma
norm = mpl.colors.Normalize(vmin=min(ION_Tvir_MIN_VALUES), vmax=max(ION_Tvir_MIN_VALUES))

# LEFT: D_ℓ(ℓ=3000) vs z
ell_target = 3000
for tvir in sorted(ION_Tvir_MIN_VALUES):
    z_plot, D_plot, E_plot = [], [], []
    for (tv, z), stats in final_stats.items():
        if tv == tvir:
            z_plot.append(z)
            D_plot.append(stats['D_mean'])
            E_plot.append(stats['D_total_err'])
    
    if len(z_plot) > 0:
        color = cmap(norm(tvir))
        ax_dl.errorbar(z_plot, D_plot, yerr=E_plot, color=color, lw=2.2, 
                       alpha=0.9, marker='o', markersize=5, capsize=3,
                       label=f'Tvir={tvir:.2f}')

ax_dl.set_xlabel(r'Redshift $z$')
ax_dl.set_ylabel(r'$D_\ell(\ell=3000)$ [$\mu\mathrm{K}^2 \cdot \mathrm{mK}$]')
ax_dl.axhline(0, color='black', ls='--', lw=1)
ax_dl.invert_xaxis()
ax_dl.grid(True, alpha=0.3)
ax_dl.legend(title='ION_Tvir_MIN', fontsize=11)

# MIDDLE: Reionization History
for tvir in ION_Tvir_MIN_VALUES:
    for (s, tv), lc in lightcones.items():
        if tv == tvir and lc is not None:
            z_nodes = np.asarray(lc.node_redshifts[::-1])
            x_e = 1.0 - np.asarray(lc.global_xH[::-1])
            color = cmap(norm(tvir))
            ax_xe.plot(z_nodes, x_e, color=color, lw=2.0, alpha=0.85)
            break

ax_xe.set_xlabel(r'Redshift $z$')
ax_xe.set_ylabel(r'Ionization Fraction $x_e$')
ax_xe.set_ylim(-0.05, 1.05)
ax_xe.invert_xaxis()
ax_xe.grid(True, alpha=0.3)

# RIGHT: Milestones
milestones = {}
for tvir in ION_Tvir_MIN_VALUES:
    for (s, tv), lc in lightcones.items():
        if tv == tvir and lc is not None:
            z_nodes = np.asarray(lc.node_redshifts[::-1])
            xe_nodes = 1.0 - np.asarray(lc.global_xH[::-1])
            z10 = z_nodes[np.argmin(np.abs(xe_nodes - 0.1))]
            z50 = z_nodes[np.argmin(np.abs(xe_nodes - 0.5))]
            z90 = z_nodes[np.argmin(np.abs(xe_nodes - 0.9))]
            milestones[tvir] = {'z10': z10, 'z50': z50, 'z90': z90}
            break

tvirs = sorted(milestones.keys())
ax_mil.plot([milestones[t]['z50'] for t in tvirs], [milestones[t]['z10'] for t in tvirs],
            'o-', color='gold', label=r'$z_{10}$')
ax_mil.plot([milestones[t]['z50'] for t in tvirs], [milestones[t]['z50'] for t in tvirs],
            's-', color='limegreen', label=r'$z_{50}$')
ax_mil.plot([milestones[t]['z50'] for t in tvirs], [milestones[t]['z90'] for t in tvirs],
            '^--', color='crimson', label=r'$z_{90}$')

ax_mil.set_xlabel(r'Reionization Midpoint $z_{50}$')
ax_mil.set_ylabel('Milestone Redshift')
ax_mil.legend()
ax_mil.grid(True, alpha=0.3)

# Colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=[ax_dl, ax_xe, ax_mil], pad=0.02, aspect=40).set_label(
    r'ION\_Tvir\_MIN [log$_{10}$ K]')

fig.suptitle(f'kSZ²–21cm at $\\ell=3000$ — Reionization History Scan (HII_EFF={HII_EFF_FACTOR_FIXED})',
             fontsize=18, fontweight='bold')

plot_name = "final_Dl3000_xe_milestones"
fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved main result: {plot_name}")
plt.close(fig)
print("\n" + "="*70)
print("FINAL SCIENCE PLOT COMPLETE!")
print("="*70)


VISUALIZING kSZ²-21cm CROSS-CORRELATION - ION_Tvir_MIN SCAN
Created final plots directory: 20DEC2025_kSZ2_21cm_ION_Tvir_scan/plots/plot_final_cell

=== CONVERTING TO ℓ-SPACE FOR ALL ION_Tvir_MIN ===
  Tvir=2.000: Converted 154 redshifts
  Tvir=2.184: Converted 154 redshifts
  Tvir=2.368: Converted 154 redshifts
  Tvir=2.553: Converted 154 redshifts
  Tvir=2.737: Converted 154 redshifts
  Tvir=2.921: Converted 154 redshifts
  Tvir=3.105: Converted 154 redshifts
  Tvir=3.289: Converted 154 redshifts
  Tvir=3.474: Converted 154 redshifts
  Tvir=3.658: Converted 154 redshifts
  Tvir=3.842: Converted 154 redshifts
  Tvir=4.026: Converted 154 redshifts
  Tvir=4.211: Converted 154 redshifts
  Tvir=4.395: Converted 154 redshifts
  Tvir=4.579: Converted 154 redshifts
  Tvir=4.763: Converted 154 redshifts
  Tvir=4.947: Converted 154 redshifts
  Tvir=5.132: Converted 154 redshifts
  Tvir=5.316: Converted 154 redshifts
  Tvir=5.500: Converted 154 redshifts

Total ION_Tvir_MIN values with ℓ-space 